# ECA-0.2 — Counterfactual Adaptive Consolidation

This notebook trains the ECA byte-stream model with a learned two-action controller: **hold** the slow state or **consolidate** the fast state into it. It is not a Transformer and uses raw UTF-8 bytes plus one structural EOS control symbol.

During training, both actions are evaluated against the observed next-byte target. The lower-loss action supplies a counterfactual teaching signal to the controller. At inference, the controller sees only causal state and previous prediction surprise; it never sees the next target. There is no fixed update interval or target consolidation percentage.

The recurrent state carries across arbitrary input chunks. `sequence_length` is only a truncated-backpropagation segment for GPU memory management, not an inference context window. This experiment tests whether the controller learns useful hold/update decisions; it does not establish reasoning or intelligence.

In [ ]:
import os, sys, torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, then rerun.')

In [ ]:
REPO = '/content/eca'
BRANCH = 'eca-0.2-counterfactual-controller'
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH https://github.com/swasthik-s/eca.git $REPO
%cd /content/eca
!git checkout $BRANCH
!pip -q install -e '.[train,dev]'

## 1. Verify the architecture before training

The tests check UTF-8 round-tripping, arbitrary chunk lengths, state carry-over, causal prefix invariance, and absence of attention modules.

In [ ]:
## 1. Verify the architecture before training

The tests cover UTF-8 round-tripping, arbitrary chunk lengths, recurrent state continuity, causal prefix invariance, a genuine hold action, counterfactual policy supervision and absence of attention modules.

## 2. Persist checkpoints to Google Drive

Colab's local disk is temporary. Mount Drive so the trained checkpoint survives runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT = '/content/drive/MyDrive/ECA/checkpoints/eca_adaptive_text.pt'
os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
print('Checkpoint path:', CHECKPOINT)

## 3. Train on Hugging Face FineWeb-Edu (streaming)

The dataset is streamed; the full corpus is not downloaded. Start with a small byte budget and short run to establish that the pipeline works. Once GPU memory and throughput are known, increase `MAX_BYTES` and `STEPS`. The initial run is intentionally a prototype-scale experiment, not GPT-1-scale pretraining.

In [ ]:
MAX_BYTES = 1_500_000
STEPS = 300
BATCH_SIZE = 4
TBPTT_SEGMENT = 256  # training memory budget only; not an inference context window
HIDDEN_SIZE = 128
EMBEDDING_SIZE = 32
!python -m eca.training.train --dataset hf --dataset-id HuggingFaceFW/fineweb-edu --dataset-config sample-10BT --documents 3000 --max-bytes $MAX_BYTES --steps $STEPS --sequence-length $TBPTT_SEGMENT --batch-size $BATCH_SIZE --hidden-size $HIDDEN_SIZE --embedding-size $EMBEDDING_SIZE --policy-loss-weight 0.25 --log-every 25 --output "$CHECKPOINT"

## 4. Inspect the trained model and generate a continuation

The EOS output class is the learned generation-stop decision. The hold/consolidate policy is a separate decision. The generation cap is a safety guard, not a fixed context length.

In [ ]:
from eca.text import ECAConfig, ECAStreamModel
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
config = ECAConfig(**checkpoint['config'])
model = ECAStreamModel(config).to(device)
model.load_state_dict(checkpoint['model'])
model.eval()
print('Parameters:', sum(p.numel() for p in model.parameters()))
print('Training step:', checkpoint['step'])
prompt = 'ECA is an experimental cognitive architecture that'
continuation = model.generate(prompt, max_new_bytes=300, temperature=0.8, device=device)
print(prompt + continuation.decode('utf-8', errors='replace'))

## 5. Record the first experiment

Do not judge the model by one generated paragraph. Record validation bits-per-byte (BPB), training loss, boundary rate, tokens/bytes per second, peak VRAM, and checkpoint size. Compare against a byte-GRU baseline with matched parameter count and training budget before claiming the adaptive mechanism helps.

**Important current limitations:** this first trainer uses truncated backpropagation through time, a small prototype corpus budget, and a simple straight-through boundary gate. The slow state is carried between training segments; long-horizon retention still needs dedicated evaluation. The model is not yet a general reasoning system.